# March 18 CE-Only Pre-Gate Benchmark

This notebook benchmarks the current pre-events periodicity gate that routes confident periodic candidates away from the stochastic `events.py` branch and into the current phase-template periodic branch.

Use it to:
- run the exact CE-only `apply_pre_periodicity_gate` logic used by `malca detect`
- benchmark the canonical periodic branch runner (`malca.events --baseline-func phase_template`)
- inspect the current split periodic post-filter structure on branch outputs
- inspect the current `periodic` and `non_periodic` pregate buckets before hardening defaults


In [2]:
from pathlib import Path
import importlib
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (missing pyproject.toml).")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from malca.config import MAG_BINS as ALL_MAG_BINS, WORKERS as DEFAULT_WORKERS
import malca.filter as malca_filter
import malca.periodicity_gate as malca_periodicity_gate
from malca.manifest import build_manifest
from malca.notebook_paths import infer_run_dir, localize_lightcurve_frame_paths, resolve_repo_path

malca_periodicity_gate = importlib.reload(malca_periodicity_gate)
malca_filter = importlib.reload(malca_filter)
apply_pre_periodicity_gate = malca_periodicity_gate.apply_pre_periodicity_gate
apply_filters = malca_filter.apply_filters

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)

print("Reloaded local MALCA modules from disk for this notebook session.")


Reloaded local MALCA modules from disk for this notebook session.


In [3]:
# Point INPUT_TABLE at an existing manifest/tag parquet with dat_path,
# or leave it as None and build a flat-directory manifest from FLAT_LC_DIR.
# Default is all configured MALCA magnitude bins.
import hashlib
import json

INPUT_TABLE = None
RUN_NAME = "runs_march18_bundle_all"
RUN_DIR = REPO_ROOT / "output" / "runs" / RUN_NAME
FLAT_LC_DIR = RUN_DIR / "bundle_assets" / "lightcurves"
INDEX_FILE = None
MAG_BINS = list(ALL_MAG_BINS)
N_WORKERS = 8
RUN_PERIODIC_BRANCH = True
RUN_PERIODIC_FILTERS = True
EVENTS_PROGRESS_CHUNK_SIZE = 250
PROGRESS_MONITOR_SECONDS = 30

INPUT_TABLE = resolve_repo_path(INPUT_TABLE, repo_root=REPO_ROOT)
FLAT_LC_DIR = resolve_repo_path(FLAT_LC_DIR, repo_root=REPO_ROOT)
INDEX_FILE = resolve_repo_path(INDEX_FILE, repo_root=REPO_ROOT)

OUTPUT_DIR = REPO_ROOT / "output" / "diagnostics" / "march18_periodicity_pregate"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The pregate is now CE-only: period grid -> {P/2, P, 2P} -> folded scatter choice.
GATE_KWARGS = {
    "min_period": 0.2,
    "max_period": 100.0,
    "n_periods": 5000,
    "ce_snr_threshold": 10.0,
    "min_points": 50,
    "scatter_ratio_max": 0.9,
    "workers": N_WORKERS,
}

# Mirror the current split post-filter structure from malca.filter/apply_filters.
# External catalog / Gaia validations are off by default here so the notebook
# stays runnable without network-bound lookups; flip them on when benchmarking
# full home-stage behavior.
PERIODIC_FILTER_STRUCTURE = {
    "core_filters": {
        "apply_evidence_strength": True,
        "min_bayes_factor": 10.0,
        "require_finite_local_bf": True,
        "apply_significant_detection": True,
        "significant_require_flag": True,
        "significant_min_peak_count": 1,
        "significant_min_run_count": 1,
        "apply_run_robustness": True,
        "min_run_count": 1,
        "max_run_count": None,
        "min_run_points": 2,
        "min_run_cameras": 2,
        "apply_morphology": False,
        "dip_morphology": "gaussian",
        "jump_morphology": "paczynski",
        "min_delta_bic": 10.0,
        "apply_score": True,
        "min_dip_score": None,
        "min_jump_score": None,
        "min_score": 0.0,
    },
    "periodicity_validation": {
        "apply_periodicity_validation": True,
        "periodicity_n_bootstrap": 50, ### EDITED FROM 1000, THIS IS FOR NOTEBOOK PURPOSS ONLY
        "periodicity_significance": 0.01,
        "periodicity_pdm_method": "plavchan",
        "periodicity_exclude_aliases": True,
        "periodicity_flag_only": True,
        "periodicity_workers": N_WORKERS,
        "periodicity_checkpoint_dir": str(OUTPUT_DIR / "periodicity_validation_checkpoints"),
        "periodicity_all_candidates": False,
        "phase_plot_max_sig": 0.01,
        "phase_plot_min_power": 0.3,
        "phase_plot_allow_alias": False,
    },
    "external_validations": {
        "apply_periodic_catalog_validation": False,
        "periodic_catalog_max_sep": 3.0,
        "periodic_catalog_flag_only": True,
        "apply_gaia_ruwe_validation": False,
        "gaia_max_ruwe": 1.4,
        "gaia_flag_only": True,
        "apply_gaia_pm_validation": False,
        "gaia_max_pm": 100.0,
        "gaia_pm_flag_only": True,
    },
}

FILTER_KWARGS = {
    key: value
    for section in PERIODIC_FILTER_STRUCTURE.values()
    for key, value in section.items()
}
FILTER_KWARGS.update({"show_tqdm": True, "verbose": False})

GATE_LOGIC_TAG = "ce_only_folded_scatter_v2"
FILTER_LOGIC_TAG = "split_periodic_filters_v1"
GATE_RUN_TAG = hashlib.md5(
    json.dumps({"logic_tag": GATE_LOGIC_TAG, **GATE_KWARGS}, sort_keys=True).encode("utf-8")
).hexdigest()[:10]
FILTER_RUN_TAG = hashlib.md5(
    json.dumps({"logic_tag": FILTER_LOGIC_TAG, **FILTER_KWARGS}, sort_keys=True).encode("utf-8")
).hexdigest()[:10]
print(f"Using gate run tag: {GATE_RUN_TAG}")
print(f"Using filter run tag: {FILTER_RUN_TAG}")


Using gate run tag: 522f4544b9
Using filter run tag: b087902da0


In [4]:
if INPUT_TABLE is not None:
    input_path = Path(INPUT_TABLE)
    if input_path.suffix.lower() in {".parquet", ".pq"}:
        df_input = pd.read_parquet(input_path)
    else:
        df_input = pd.read_csv(input_path)

    input_run_dir = infer_run_dir(input_path) or infer_run_dir(RUN_DIR)
    df_input, localized_counts = localize_lightcurve_frame_paths(
        df_input,
        run_dir=input_run_dir,
        repo_root=REPO_ROOT,
        path_columns=("dat_path", "path", "lc_path"),
    )
    if localized_counts:
        print(f"Localized stored light-curve paths: {localized_counts}")

    input_path_col = "dat_path" if "dat_path" in df_input.columns else ("path" if "path" in df_input.columns else None)
    if input_path_col is not None:
        exists_mask = df_input[input_path_col].map(lambda x: Path(str(x)).expanduser().exists() if pd.notna(x) else False)
        if not bool(exists_mask.all()):
            print(f"Retained {int(exists_mask.sum()):,} rows with bundled local light curves.")
        df_input = df_input[exists_mask].reset_index(drop=True)
else:
    df_input = build_manifest(
        None,
        None,
        mag_bins=MAG_BINS,
        id_column="asas_sn_id",
        flat_lc_dir=FLAT_LC_DIR,
        index_file=INDEX_FILE,
        show_progress=True,
        n_workers=N_WORKERS,
    )
    df_input = df_input[df_input["dat_exists"]].reset_index(drop=True)

print(f"Loaded {len(df_input):,} candidates")
display(df_input.head())


flat light curves: 100%|██████████| 9569/9569 [00:00<00:00, 197880.42it/s]

[warn] 9569 flat light curves lacked mag_bin metadata; downstream mag-bin filters will treat them as unparsed.
Loaded 9,569 candidates


,source_id,mag_bin,index_num,index_csv,lc_dir,lc_dir_exists,dat_path,dat_exists
0,103079222309,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True
1,103079223768,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True
2,103079224553,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True
3,103079238346,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True
4,103079246967,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True


In [5]:
checkpoint_path = OUTPUT_DIR / f"march18_periodicity_gate_checkpoint_{GATE_RUN_TAG}.parquet"
df_gate = apply_pre_periodicity_gate(
    df_input,
    path_col="dat_path" if "dat_path" in df_input.columns else "path",
    checkpoint_path=checkpoint_path,
    show_tqdm=True,
    **GATE_KWARGS,
)
output_file = OUTPUT_DIR / f"march18_periodicity_gate_{GATE_RUN_TAG}.parquet"
df_gate.to_parquet(output_file, index=False)
print(f"Saved gate output to {output_file}")

label_counts = df_gate["pre_periodicity_label"].value_counts(dropna=False).rename_axis("label").to_frame("n")
display(label_counts)

print("Top-level summary below uses only the active CE-only gate metrics.")
summary_cols = [
    "pre_periodicity_score",
    "pre_periodicity_scatter_ratio",
]
summary_cols = [col for col in summary_cols if col in df_gate.columns]
display(df_gate.groupby("pre_periodicity_label")[summary_cols].median(numeric_only=True))


[pre_periodicity_gate] 8126 cached, processing 1443 light curves


Pre-periodicity gate: 100%|██████████| 1443/1443 [02:36<00:00,  9.21it/s]


Saved gate output to /home/calder/code/malca/output/diagnostics/march18_periodicity_pregate/march18_periodicity_gate_522f4544b9.parquet


,n
label,
non_periodic,7012
periodic,2557


Top-level summary below uses only the active CE-only gate metrics.


,pre_periodicity_score,pre_periodicity_scatter_ratio
pre_periodicity_label,,
non_periodic,7.620272,0.797872
periodic,13.008705,0.773748


In [ ]:
import threading
import time


def _safe_count_text_lines(path):
    path = Path(path)
    if not path.exists():
        return 0
    try:
        with path.open("r", encoding="utf-8", errors="ignore") as handle:
            return sum(1 for line in handle if line.strip())
    except Exception:
        return None


def _safe_count_parquet_rows(path):
    path = Path(path)
    if not path.exists():
        return 0
    try:
        return int(len(pd.read_parquet(path, columns=["path"])))
    except Exception:
        return None


def _start_progress_monitor(label, path, *, kind="lines", target=None, interval=None):
    interval = float(PROGRESS_MONITOR_SECONDS if interval is None else interval)
    path = Path(path)
    stop_event = threading.Event()
    count_fn = _safe_count_parquet_rows if kind == "parquet_rows" else _safe_count_text_lines

    def _format_count(value):
        if value is None:
            return "temporarily unreadable"
        if target is None:
            return f"{value:,}"
        pct = 100.0 * float(value) / float(max(int(target), 1))
        return f"{value:,}/{int(target):,} ({pct:.1f}%)"

    def _monitor():
        last_value = object()
        while not stop_event.wait(interval):
            value = count_fn(path)
            if value != last_value:
                print(f"[{label}] {time.strftime('%H:%M:%S')} {path.name}: {_format_count(value)}", flush=True)
                last_value = value

    initial = count_fn(path)
    print(f"[{label}] monitoring {path}: {_format_count(initial)}", flush=True)
    thread = threading.Thread(target=_monitor, daemon=True)
    thread.start()
    return stop_event, thread


def _stop_progress_monitor(stop_event, thread):
    if stop_event is not None:
        stop_event.set()
    if thread is not None:
        thread.join(timeout=1.0)


if RUN_PERIODIC_BRANCH:
    df_periodic_input = df_gate[df_gate["pre_periodic_flag"]].copy()
    print(f"Periodic branch input rows: {len(df_periodic_input):,}")
    periodic_output = OUTPUT_DIR / f"march18_periodic_branch_events_{GATE_RUN_TAG}.parquet"
    periodic_paths_file = OUTPUT_DIR / f"march18_periodic_branch_paths_{GATE_RUN_TAG}.txt"
    periodic_metadata_file = OUTPUT_DIR / f"march18_periodic_branch_metadata_{GATE_RUN_TAG}.csv"
    periodic_filtered_output = OUTPUT_DIR / f"march18_periodic_branch_filtered_{FILTER_RUN_TAG}.parquet"

    filter_structure_rows = [
        {"section": section_name, "kwarg": key, "value": value}
        for section_name, section in PERIODIC_FILTER_STRUCTURE.items()
        for key, value in section.items()
    ]
    print("Current split periodic filter structure")
    display(pd.DataFrame(filter_structure_rows))

    if df_periodic_input.empty:
        df_periodic_events = pd.DataFrame()
        df_periodic_filtered = pd.DataFrame()
        print("No confident periodic candidates routed to the periodic branch")
    else:
        path_col = "dat_path" if "dat_path" in df_periodic_input.columns else "path"
        metadata_cols = [path_col]
        for col in ["source_id", "asas_sn_id", "name", "mag_bin"]:
            if col in df_periodic_input.columns and col not in metadata_cols:
                metadata_cols.append(col)
        if "excluded_cameras" in df_periodic_input.columns and "excluded_cameras" not in metadata_cols:
            metadata_cols.append("excluded_cameras")
        for col in [
            "pre_periodicity_label",
            "pre_periodic_flag",
            "pre_periodicity_selected_period",
            "pre_periodicity_method",
        ]:
            if col in df_periodic_input.columns and col not in metadata_cols:
                metadata_cols.append(col)

        df_periodic_metadata = df_periodic_input[metadata_cols].rename(columns={path_col: "path"}).copy()
        df_periodic_metadata.to_csv(periodic_metadata_file, index=False)
        periodic_paths_file.write_text(
            "".join(f"{path_value}\n" for path_value in df_periodic_metadata["path"].astype(str)),
            encoding="utf-8",
        )
        event_chunk_size = max(1, min(int(EVENTS_PROGRESS_CHUNK_SIZE), len(df_periodic_metadata)))
        events_processed_log = periodic_output.with_name(f"{periodic_output.stem}_PROCESSED.txt")

        cmd = [
            sys.executable,
            "-m",
            "malca.events",
            "--input-file",
            str(periodic_paths_file),
            "--metadata-csv",
            str(periodic_metadata_file),
            "--output",
            str(periodic_output),
            "--output-format",
            "parquet",
            "--baseline-func",
            "phase_template",
            "--workers",
            str(N_WORKERS),
            "--chunk-size",
            str(event_chunk_size),
            "--verbose",
            "--overwrite",
        ]
        print("Running canonical periodic branch command:")
        print(" ".join(cmd))
        print(f"Event stage progress: {len(df_periodic_metadata):,} light curves, {N_WORKERS} workers, chunk_size={event_chunk_size}")
        events_stop, events_thread = _start_progress_monitor(
            "events processed log",
            events_processed_log,
            kind="lines",
            target=len(df_periodic_metadata),
        )
        try:
            subprocess.run(cmd, check=True)
        finally:
            _stop_progress_monitor(events_stop, events_thread)
        events_done = _safe_count_text_lines(events_processed_log)
        print(
            f"[events processed log] final rows: {events_done:,}/{len(df_periodic_metadata):,}"
            if events_done is not None
            else "[events processed log] final rows: unreadable"
        )

        df_periodic_events = pd.read_parquet(periodic_output)
        print(f"Saved canonical periodic-branch output to {periodic_output}")
        event_preview_cols = [
            "source_id",
            "mag_bin",
            "baseline_source",
            "dip_significant",
            "jump_significant",
            "dip_best_morph",
            "dip_best_amp",
            "dip_max_run_points",
            "dipper_score",
            "jumper_score",
            "pre_periodicity_selected_period",
            "pre_periodicity_method",
        ]
        event_preview_cols = [col for col in event_preview_cols if col in df_periodic_events.columns]
        event_preview = df_periodic_events[event_preview_cols].copy()
        event_sort_cols = [col for col in ["dipper_score", "dip_best_amp"] if col in event_preview.columns]
        if event_sort_cols:
            event_preview = event_preview.sort_values(event_sort_cols, ascending=[False] * len(event_sort_cols))
        display(event_preview.head(25))

        if RUN_PERIODIC_FILTERS:
            periodicity_checkpoint_dir = FILTER_KWARGS.get("periodicity_checkpoint_dir")
            periodicity_checkpoint_file = (
                Path(periodicity_checkpoint_dir) / "lsp_checkpoint.parquet"
                if periodicity_checkpoint_dir and FILTER_KWARGS.get("apply_periodicity_validation")
                else None
            )
            print(f"Running periodic branch filters on {len(df_periodic_events):,} event rows")
            filter_stop = filter_thread = None
            if periodicity_checkpoint_file is not None:
                print("The apply_filters bar tracks filter stages; this monitor tracks periodicity-validation checkpoint growth.")
                filter_stop, filter_thread = _start_progress_monitor(
                    "periodicity checkpoint",
                    periodicity_checkpoint_file,
                    kind="parquet_rows",
                )
            try:
                df_periodic_filtered = apply_filters(df_periodic_events, **FILTER_KWARGS)
            finally:
                _stop_progress_monitor(filter_stop, filter_thread)
            if periodicity_checkpoint_file is not None:
                checkpoint_rows = _safe_count_parquet_rows(periodicity_checkpoint_file)
                print(f"[periodicity checkpoint] final rows: {checkpoint_rows:,}" if checkpoint_rows is not None else "[periodicity checkpoint] final rows: unreadable")
            df_periodic_filtered.to_parquet(periodic_filtered_output, index=False)
            print(f"Saved periodic-branch filtered output to {periodic_filtered_output}")

            failed_cols = [col for col in df_periodic_filtered.columns if col.startswith("failed_")]
            filter_summary = (
                pd.Series(
                    {
                        col: int(df_periodic_filtered[col].fillna(False).astype(bool).sum())
                        for col in failed_cols
                    }
                )
                .rename_axis("filter")
                .to_frame("n_failed")
                .sort_values("n_failed", ascending=False)
            )

            core_failed_cols = [
                col
                for col in [
                    "failed_posterior_strength",
                    "failed_significant_detection",
                    "failed_run_robustness",
                    "failed_morphology",
                    "failed_score",
                ]
                if col in df_periodic_filtered.columns
            ]
            periodicity_checked_cols = [
                col
                for col in ["lsp_period", "periodicity_score", "periodic_flag"]
                if col in df_periodic_filtered.columns
            ]
            periodicity_eligible = (
                ~df_periodic_filtered[core_failed_cols].fillna(False).any(axis=1)
                if core_failed_cols
                else pd.Series(True, index=df_periodic_filtered.index, dtype=bool)
            )
            periodicity_checked = (
                df_periodic_filtered[periodicity_checked_cols].notna().any(axis=1)
                if periodicity_checked_cols
                else pd.Series(False, index=df_periodic_filtered.index, dtype=bool)
            )
            filter_audit = pd.DataFrame(
                [
                    {
                        "n_rows": int(len(df_periodic_filtered)),
                        "periodicity_validation_eligible": int(periodicity_eligible.sum()),
                        "periodicity_validation_checked": int(periodicity_checked.sum()),
                        "failed_any": int(
                            df_periodic_filtered.get(
                                "failed_any",
                                pd.Series(False, index=df_periodic_filtered.index, dtype=bool),
                            ).fillna(False).sum()
                        ),
                    }
                ]
            )

            print("Periodic branch filter audit")
            print("Downstream `periodicity_*` columns below come from post-filter periodic validation, not from the CE-only pregate.")
            display(filter_audit)
            display(filter_summary.head(20))

            filtered_preview_cols = [
                "source_id",
                "mag_bin",
                "failed_any",
                "failed_posterior_strength",
                "failed_significant_detection",
                "failed_run_robustness",
                "failed_score",
                "failed_periodicity",
                "periodicity_score",
                "periodic_flag",
                "phase_plot_ready",
                "lsp_period",
                "pre_periodicity_label",
                "pre_periodicity_selected_period",
            ]
            filtered_preview_cols = [col for col in filtered_preview_cols if col in df_periodic_filtered.columns]
            filtered_preview = df_periodic_filtered[filtered_preview_cols].copy()
            filtered_sort_cols = [col for col in ["failed_any", "periodicity_score", "dipper_score"] if col in filtered_preview.columns]
            filtered_sort_ascending = [True if col == "failed_any" else False for col in filtered_sort_cols]
            if filtered_sort_cols:
                filtered_preview = filtered_preview.sort_values(filtered_sort_cols, ascending=filtered_sort_ascending)
            display(filtered_preview.head(25))
        else:
            df_periodic_filtered = pd.DataFrame()
            print("RUN_PERIODIC_FILTERS is False; skipping current periodic post-filter benchmark")
else:
    df_periodic_events = pd.DataFrame()
    df_periodic_filtered = pd.DataFrame()
    print("RUN_PERIODIC_BRANCH is False; skipping periodic branch benchmark")


Periodic branch input rows: 2,557
Current split periodic filter structure


,section,kwarg,value
0,core_filters,apply_evidence_strength,True
1,core_filters,min_bayes_factor,10.0
2,core_filters,require_finite_local_bf,True
3,core_filters,apply_significant_detection,True
4,core_filters,significant_require_flag,True
5,core_filters,significant_min_peak_count,1
6,core_filters,significant_min_run_count,1
7,core_filters,apply_run_robustness,True
8,core_filters,min_run_count,1
9,core_filters,max_run_count,None


Running canonical periodic branch command:
/home/calder/miniforge3/envs/malca/bin/python -m malca.events --input-file /home/calder/code/malca/output/diagnostics/march18_periodicity_pregate/march18_periodic_branch_paths_522f4544b9.txt --metadata-csv /home/calder/code/malca/output/diagnostics/march18_periodicity_pregate/march18_periodic_branch_metadata_522f4544b9.csv --output /home/calder/code/malca/output/diagnostics/march18_periodicity_pregate/march18_periodic_branch_events_522f4544b9.parquet --output-format parquet --baseline-func phase_template --workers 8 --chunk-size 250 --verbose --overwrite
Event stage progress: 2,557 light curves, 8 workers, chunk_size=250
[events processed log] monitoring /home/calder/code/malca/output/diagnostics/march18_periodicity_pregate/march18_periodic_branch_events_522f4544b9_PROCESSED.txt: 3,920/2,557 (153.3%)
Overwriting checkpoint log: /home/calder/code/malca/output/diagnostics/march18_periodicity_pregate/march18_periodic_branch_events_522f4544b9_PROC

LCs:   5%|▌         | 140/2557 [00:26<07:05,  5.68lc/s]

[events processed log] 11:40:10 march18_periodic_branch_events_522f4544b9_PROCESSED.txt: 0/2,557 (0.0%)


...

LCs:  29%|██▉       | 747/2557 [02:42<07:30,  4.02lc/s]

Wrote chunk: 250 rows (total: 750) (dip_sig=520, jump_sig=85, any_sig=533)


LCs:  32%|███▏      | 822/2557 [02:56<04:28,  6.46lc/s]

[events processed log] 11:42:40 march18_periodic_branch_events_522f4544b9_PROCESSED.txt: 750/2,557 (29.3%)


LCs:  34%|███▍      | 866/2557 [03:06<06:39,  4.23lc/s]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
label_colors = {
    "periodic": "tab:green",
    "non_periodic": "tab:red",
}

threshold_ce_snr = float(GATE_KWARGS["ce_snr_threshold"])
threshold_scatter = float(GATE_KWARGS["scatter_ratio_max"])

df_gate["pre_periodicity_label"].value_counts().plot.bar(ax=axes[0, 0], title="Gate Labels")
axes[0, 0].set_ylabel("count")

df_gate["pre_periodicity_score"].dropna().plot.hist(ax=axes[0, 1], bins=40, title="Selected-Method SNR")
axes[0, 1].set_xlabel("pre_periodicity_score")
axes[0, 1].axvline(threshold_ce_snr, color="0.35", ls="--", lw=1)

df_gate["pre_periodicity_scatter_ratio"].dropna().plot.hist(ax=axes[1, 0], bins=40, title="Folded Scatter Ratio")
axes[1, 0].set_xlabel("folded/raw scatter")
axes[1, 0].axvline(threshold_scatter, color="0.35", ls="--", lw=1)

for label, color in label_colors.items():
    subset = df_gate[df_gate["pre_periodicity_label"] == label]
    if subset.empty:
        continue
    axes[1, 1].scatter(
        subset["pre_ce_snr"],
        subset["pre_periodicity_scatter_ratio"],
        s=8,
        alpha=0.5,
        color=color,
        label=label,
    )
axes[1, 1].axvline(threshold_ce_snr, color="0.35", ls="--", lw=1)
axes[1, 1].axhline(threshold_scatter, color="0.35", ls="--", lw=1)
axes[1, 1].set_title("CE SNR vs Folded Scatter")
axes[1, 1].set_xlabel("pre_ce_snr")
axes[1, 1].set_ylabel("pre_periodicity_scatter_ratio")
axes[1, 1].legend(frameon=False)
axes[1, 1].grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
cols = [
    "source_id",
    "mag_bin",
    "pre_periodicity_label",
    "pre_periodicity_method",
    "pre_periodicity_selected_period",
    "pre_periodicity_score",
    "pre_periodicity_scatter_ratio",
    "pre_periodicity_support_count",
    "pre_ce_snr",
    "pre_ce_entropy",
    "pre_periodicity_reason",
]
cols = [col for col in cols if col in df_gate.columns]

print("Top confident periodic candidates")
display(df_gate[df_gate["pre_periodic_flag"]].sort_values("pre_periodicity_score", ascending=False)[cols].head(50))

print("Top non-periodic near-misses")
display(
    df_gate[~df_gate["pre_periodic_flag"]]
    .sort_values(["pre_periodicity_score", "pre_periodicity_scatter_ratio"], ascending=[False, True], na_position="last")[cols]
    .head(50)
)


In [ ]:
# Edge-case periodicity candidates across CE-only gate labels
import numpy as np

_gate_globals = apply_pre_periodicity_gate.__globals__
load_lightcurve_df = _gate_globals["load_lightcurve_df"]
clean_lc = _gate_globals["clean_lc"]
apply_simple_band_median_offset = _gate_globals["apply_simple_band_median_offset"]
BAD_CAMERA_SCATTER_RATIO_THRESHOLD = _gate_globals["BAD_CAMERA_SCATTER_RATIO_THRESHOLD"]
CLEAN_LC_MAX_ERROR_ABSOLUTE = _gate_globals["CLEAN_LC_MAX_ERROR_ABSOLUTE"]
CLEAN_LC_MAX_ERROR_SIGMA = _gate_globals["CLEAN_LC_MAX_ERROR_SIGMA"]

EDGE_CASE_PATH_COL = "dat_path" if "dat_path" in df_gate.columns else "path"
EDGE_CASE_LABEL_COL = "pre_periodicity_label"
EDGE_CASE_SCORE_COL = "pre_periodicity_score"
EDGE_CASE_PERIOD_COL = "pre_periodicity_selected_period"
EDGE_CASE_ID_COL = next((col for col in ["source_id", "asas_sn_id", "name"] if col in df_gate.columns), None)
EDGE_CASE_DISTANCE_COL = "pre_periodicity_edge_distance"
EDGE_CASE_STRENGTH_COL = "pre_periodicity_edge_strength"

EDGE_CASE_SPECS = [
    ("periodic", "Most periodic periodic", "closest"),
    ("periodic", "Least periodic periodic", "farthest"),
    ("non_periodic", "Closest non-periodic near miss", "closest"),
    ("non_periodic", "Least periodic non-periodic", "farthest"),
]


def _edge_num(df, col, default=np.nan):
    if col not in df.columns:
        return pd.Series(default, index=df.index, dtype=float)
    return pd.to_numeric(df[col], errors="coerce")


def add_periodicity_edge_metrics(df):
    out = df.copy()

    ce_snr_thr = float(GATE_KWARGS.get("ce_snr_threshold", 10.0))
    scatter_thr = float(GATE_KWARGS.get("scatter_ratio_max", 0.8))

    support_count = _edge_num(out, "pre_periodicity_support_count", default=0.0).fillna(0.0)
    scatter = _edge_num(out, "pre_periodicity_scatter_ratio")
    ce_snr = _edge_num(out, "pre_ce_snr")
    alias_flag = out.get("pre_periodicity_alias_flag", pd.Series(False, index=out.index)).fillna(False).astype(bool)

    ce_def = pd.Series(
        np.where(np.isfinite(ce_snr) & (ce_snr > 0), np.maximum((ce_snr_thr / ce_snr) - 1.0, 0.0), 1.0),
        index=out.index,
    )
    scatter_def = pd.Series(
        np.where(np.isfinite(scatter), np.maximum((scatter / scatter_thr) - 1.0, 0.0), 1.0),
        index=out.index,
    )
    support_def = pd.Series(np.maximum(1.0 - support_count, 0.0), index=out.index)
    alias_def = pd.Series(0.25 * alias_flag.astype(float), index=out.index)

    ce_strength = pd.Series(
        np.where(np.isfinite(ce_snr), np.clip(ce_snr / ce_snr_thr, 0.0, 5.0), 0.0),
        index=out.index,
    )
    scatter_strength = pd.Series(
        np.where(np.isfinite(scatter) & (scatter > 0), np.clip(scatter_thr / scatter, 0.0, 5.0), 0.0),
        index=out.index,
    )

    out[EDGE_CASE_DISTANCE_COL] = ce_def + scatter_def + support_def + alias_def
    out[EDGE_CASE_STRENGTH_COL] = (
        support_count
        + ce_strength
        + (0.50 * scatter_strength)
        - (0.50 * alias_flag.astype(float))
    )
    return out


def _pick_edge_case(subset, which):
    if which == "closest":
        ranked = subset.sort_values(
            [EDGE_CASE_DISTANCE_COL, EDGE_CASE_STRENGTH_COL, EDGE_CASE_SCORE_COL],
            ascending=[True, False, False],
            na_position="last",
        )
    else:
        ranked = subset.sort_values(
            [EDGE_CASE_DISTANCE_COL, EDGE_CASE_STRENGTH_COL, EDGE_CASE_SCORE_COL],
            ascending=[False, True, True],
            na_position="last",
        )
    return ranked.iloc[0].copy()


def select_periodicity_edge_cases(df, *, label_col=EDGE_CASE_LABEL_COL):
    enriched = add_periodicity_edge_metrics(df)
    rows = []
    missing = []

    for label, edge_case, which in EDGE_CASE_SPECS:
        subset = enriched[enriched[label_col] == label].copy()
        if subset.empty:
            missing.append(edge_case)
            continue

        chosen = _pick_edge_case(subset, which)
        chosen["edge_case"] = edge_case
        rows.append(chosen)

    edge_df = pd.DataFrame(rows)
    if edge_df.empty:
        print("No edge cases available for plotting")
        return edge_df

    display_cols = [
        "edge_case",
        EDGE_CASE_ID_COL,
        "mag_bin",
        "pre_periodicity_label",
        EDGE_CASE_DISTANCE_COL,
        EDGE_CASE_STRENGTH_COL,
        "pre_periodicity_method",
        "pre_periodicity_base_period",
        "pre_periodicity_selected_period",
        "pre_periodicity_harmonic_factor",
        "pre_periodicity_scatter_ratio",
        "pre_periodicity_alias_flag",
        "pre_periodicity_support_count",
        "pre_ce_snr",
        "pre_ce_entropy",
        "pre_ce_support",
        "pre_periodicity_v_minus_g_median_offset",
        "pre_periodicity_reason",
    ]
    display_cols = [col for col in display_cols if col and col in edge_df.columns]
    print("Periodicity gate edge cases")
    print("Smaller pre_periodicity_edge_distance means closer to satisfying the CE-only periodic gate.")
    display(edge_df[display_cols])
    if missing:
        print("Missing edge cases:", ", ".join(missing))
    return edge_df.reset_index(drop=True)


edge_cases = select_periodicity_edge_cases(df_gate)


In [ ]:
# Plot raw and phase-folded views for the periodicity gate edge cases
BAND_COLORS = {0: "tab:green", 1: "tab:blue"}
BAND_LABELS = {0: "g band", 1: "V band"}


def _excluded_camera_set(value):
    if isinstance(value, str):
        return {int(item.strip()) for item in value.split(",") if item.strip()}
    if isinstance(value, (set, list, tuple)):
        return {int(item) for item in value}
    return set()


def _coerce_scalar(value):
    return pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]


def _format_scalar(value, fmt):
    scalar = _coerce_scalar(value)
    return format(float(scalar), fmt) if pd.notna(scalar) else "nan"


def _load_gate_lightcurve(row):
    df_lc = load_lightcurve_df(
        row[EDGE_CASE_PATH_COL],
        filter_bad_cameras_enabled=True,
        bad_camera_scatter_ratio=BAD_CAMERA_SCATTER_RATIO_THRESHOLD,
    )
    excluded = _excluded_camera_set(row.get("excluded_cameras"))
    if excluded and "camera#" in df_lc.columns:
        df_lc = df_lc[~df_lc["camera#"].isin(excluded)].reset_index(drop=True)
    df_lc = clean_lc(
        df_lc,
        max_error_absolute=CLEAN_LC_MAX_ERROR_ABSOLUTE,
        max_error_sigma=CLEAN_LC_MAX_ERROR_SIGMA,
    )
    return df_lc.reset_index(drop=True)


def _scatter_by_band(ax, df_lc, x, *, xlabel, ylabel=None):
    if "v_g_band" in df_lc.columns:
        for band, band_df in df_lc.groupby("v_g_band"):
            band_int = int(band)
            ax.scatter(
                band_df[x],
                band_df["mag"],
                s=10,
                alpha=0.65,
                color=BAND_COLORS.get(band_int, "tab:gray"),
                label=BAND_LABELS.get(band_int, f"band {band_int}"),
            )
    else:
        ax.scatter(df_lc[x], df_lc["mag"], s=10, alpha=0.65, color="tab:gray")
    ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.invert_yaxis()
    ax.grid(alpha=0.2)


def _gate_thresholds():
    return {
        "scatter": float(GATE_KWARGS.get("scatter_ratio_max", 0.8)),
        "ce_snr": float(GATE_KWARGS.get("ce_snr_threshold", 10.0)),
    }


def _gate_blockers(row):
    thr = _gate_thresholds()
    if str(row.get("pre_periodicity_label", "")).strip() == "periodic":
        return []

    blockers = []
    ce_snr_val = _coerce_scalar(row.get("pre_ce_snr"))
    scatter_val = _coerce_scalar(row.get("pre_periodicity_scatter_ratio"))
    support_raw = _coerce_scalar(row.get("pre_periodicity_support_count"))
    support_count_val = int(support_raw) if pd.notna(support_raw) else 0
    alias_flag_val = bool(row.get("pre_periodicity_alias_flag", False))

    if (not np.isfinite(ce_snr_val)) or ce_snr_val < thr["ce_snr"]:
        blockers.append(f"ce[snr<{thr['ce_snr']:g}]")
    if not np.isfinite(scatter_val):
        blockers.append("no_folded_scatter")
    elif scatter_val > thr["scatter"]:
        blockers.append(f"scatter>{thr['scatter']:g}")
    if support_count_val < 1:
        blockers.append("support<1")
    if alias_flag_val:
        blockers.append("alias")

    return list(dict.fromkeys(blockers))


def _format_gate_blockers(row, *, width=84):
    blockers = _gate_blockers(row)
    if not blockers:
        return "reject=none"

    lines = []
    prefix = "reject="
    indent = " " * len(prefix)
    current = prefix
    for blocker in blockers:
        token = blocker if current == prefix else f", {blocker}"
        if len(current) + len(token) > width and current != prefix:
            lines.append(current)
            current = indent + blocker
        else:
            current += token
    lines.append(current)
    return "\n".join(lines)


def _plot_edge_case(ax_raw, ax_phase, row):
    df_lc = _load_gate_lightcurve(row)
    candidate_id = row.get(EDGE_CASE_ID_COL, row.name) if EDGE_CASE_ID_COL else row.name
    period = _coerce_scalar(row.get(EDGE_CASE_PERIOD_COL))
    period_str = f"{period:.5g} d" if pd.notna(period) and period > 0 else "n/a"
    base_period = _coerce_scalar(row.get("pre_periodicity_base_period"))
    base_period_str = f"{base_period:.5g} d" if pd.notna(base_period) and base_period > 0 else "n/a"
    harmonic_factor = _coerce_scalar(row.get("pre_periodicity_harmonic_factor"))
    harmonic_factor_str = f"{harmonic_factor:.5g}" if pd.notna(harmonic_factor) and harmonic_factor > 0 else "n/a"
    edge_distance = _format_scalar(row.get(EDGE_CASE_DISTANCE_COL), ".3f")
    edge_strength = _format_scalar(row.get(EDGE_CASE_STRENGTH_COL), ".2f")
    scatter = _format_scalar(row.get("pre_periodicity_scatter_ratio"), ".2f")
    ce_snr = _format_scalar(row.get("pre_ce_snr"), ".1f")
    ce_entropy = _format_scalar(row.get("pre_ce_entropy"), ".3f")
    support_count = _format_scalar(row.get("pre_periodicity_support_count"), ".0f")
    v_minus_g_offset = _format_scalar(row.get("pre_periodicity_v_minus_g_median_offset"), ".3f")
    ce_support = bool(row.get("pre_ce_support", False))
    alias_flag = bool(row.get("pre_periodicity_alias_flag", False))
    selected_method = str(row.get("pre_periodicity_method", "")).strip()
    reject_text = _format_gate_blockers(row)

    header = f"{row['edge_case']} | candidate {candidate_id}"
    meta = (
        f"label={row.get('pre_periodicity_label', 'n/a')}  distance={edge_distance}  strength={edge_strength}\n"
        f"selected={period_str}  base={base_period_str}  factor={harmonic_factor_str}\n"
        f"method={selected_method or 'n/a'}  support={support_count}  ce_support={ce_support}  alias={alias_flag}\n"
        f"v_minus_g_offset={v_minus_g_offset}\n"
        f"ce_snr={ce_snr}  ce_entropy={ce_entropy}  scatter={scatter}\n"
        f"pass_reason={row.get('pre_periodicity_reason', '')}\n"
        f"{reject_text}"
    )
    ax_raw.set_title(header, loc="left", fontsize=11)

    if df_lc.empty:
        ax_raw.text(0.5, 0.5, "No clean light curve", ha="center", va="center")
        ax_phase.text(0.5, 0.5, "No clean light curve", ha="center", va="center")
        ax_raw.axis("off")
        ax_phase.axis("off")
        return

    df_plot, _ = apply_simple_band_median_offset(df_lc)

    _scatter_by_band(ax_raw, df_plot, "JD", xlabel="JD", ylabel="mag (band-aligned)")
    ax_raw.text(
        0.01,
        0.02,
        meta,
        transform=ax_raw.transAxes,
        fontsize=8,
        va="bottom",
        ha="left",
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )
    handles, labels = ax_raw.get_legend_handles_labels()
    if handles:
        ax_raw.legend(handles, labels, loc="upper right", fontsize=8)

    if not pd.notna(period) or period <= 0:
        ax_phase.text(0.5, 0.5, "No selected period", ha="center", va="center")
        ax_phase.set_axis_off()
        return

    phase = np.mod((df_plot["JD"] - df_plot["JD"].min()) / float(period), 1.0)
    phase_df = df_plot.assign(phase=phase)
    phase_wrap = pd.concat(
        [phase_df.assign(phase_plot=phase_df["phase"]), phase_df.assign(phase_plot=phase_df["phase"] + 1.0)],
        ignore_index=True,
    )
    _scatter_by_band(ax_phase, phase_wrap, "phase_plot", xlabel="phase")
    ax_phase.set_xlim(0.0, 2.0)
    ax_phase.axvline(1.0, color="0.5", ls="--", lw=1)
    ax_phase.set_title(f"Phase-folded | sel={period_str} | base={base_period_str} | f={harmonic_factor_str}", fontsize=10)


if edge_cases.empty:
    print("No edge cases to plot")
else:
    fig, axes = plt.subplots(len(edge_cases), 2, figsize=(14, 3.8 * len(edge_cases)), constrained_layout=True)
    axes = np.atleast_2d(axes)
    for ax_pair, (_, row) in zip(axes, edge_cases.iterrows()):
        _plot_edge_case(ax_pair[0], ax_pair[1], row)
    plt.show()


In [ ]:
# Sample 50 candidates from each gate class
SAMPLE_SIZE = 50
SAMPLE_BASE_SEED = 18
SAMPLE_PLOT_BATCH_SIZE = 10
SAMPLE_ID_COL = next((col for col in ["source_id", "asas_sn_id", "name"] if col in df_gate.columns), None)

if "add_periodicity_edge_metrics" in globals():
    df_gate_samples = add_periodicity_edge_metrics(df_gate)
else:
    df_gate_samples = df_gate.copy()

SAMPLE_DISPLAY_COLS = [
    SAMPLE_ID_COL,
    "mag_bin",
    "pre_periodicity_label",
    "pre_periodicity_edge_distance",
    "pre_periodicity_edge_strength",
    "pre_periodicity_method",
    "pre_periodicity_base_period",
    "pre_periodicity_selected_period",
    "pre_periodicity_harmonic_factor",
    "pre_periodicity_score",
    "pre_periodicity_scatter_ratio",
    "pre_periodicity_alias_flag",
    "pre_periodicity_support_count",
    "pre_ce_snr",
    "pre_ce_entropy",
    "pre_ce_support",
    "pre_periodicity_v_minus_g_median_offset",
    "pre_periodicity_reason",
]
SAMPLE_DISPLAY_COLS = [col for col in SAMPLE_DISPLAY_COLS if col and col in df_gate_samples.columns]


def sample_gate_candidates(df, label, *, n=SAMPLE_SIZE, seed=SAMPLE_BASE_SEED):
    subset = df[df["pre_periodicity_label"] == label].copy()
    if subset.empty:
        print(f"No {label} candidates available")
        return subset

    n_take = min(int(n), len(subset))
    sampled = subset.sample(n=n_take, random_state=int(seed)).reset_index(drop=True)
    sort_cols = [col for col in ["pre_periodicity_edge_distance", SAMPLE_ID_COL] if col and col in sampled.columns]
    if sort_cols:
        sampled = sampled.sort_values(sort_cols, ascending=[True] * len(sort_cols), na_position="last").reset_index(drop=True)

    print(f"{label}: showing {n_take} sampled candidates out of {len(subset)}")
    display(sampled[SAMPLE_DISPLAY_COLS])
    return sampled


def plot_gate_candidate_sample(sampled, title, *, batch_size=SAMPLE_PLOT_BATCH_SIZE):
    if sampled.empty:
        print(f"No sampled candidates to plot for {title}")
        return
    if "_plot_edge_case" not in globals():
        print("Run the edge-case plotting cell first so sampled candidates can be plotted.")
        return

    sampled = sampled.reset_index(drop=True)
    for start in range(0, len(sampled), int(batch_size)):
        batch = sampled.iloc[start:start + int(batch_size)].copy().reset_index(drop=True)
        fig, axes = plt.subplots(len(batch), 2, figsize=(14, 3.8 * len(batch)), constrained_layout=True)
        axes = np.atleast_2d(axes)
        for offset, (ax_pair, (_, row)) in enumerate(zip(axes, batch.iterrows()), start=start + 1):
            row = row.copy()
            row["edge_case"] = f"{title} #{offset}"
            _plot_edge_case(ax_pair[0], ax_pair[1], row)
        plt.show()


In [ ]:
# Sample 50 periodic candidates
periodic_sample_50 = sample_gate_candidates(
    df_gate_samples,
    "periodic",
    n=SAMPLE_SIZE,
    seed=SAMPLE_BASE_SEED,
)
plot_gate_candidate_sample(periodic_sample_50, "Periodic sample")


In [ ]:
# Plot the closest non-periodic near-misses
near_miss_50 = (
    df_gate_samples[df_gate_samples["pre_periodicity_label"] == "non_periodic"]
    .sort_values(["pre_periodicity_edge_distance", "pre_periodicity_score"], ascending=[True, False], na_position="last")
    .head(SAMPLE_SIZE)
    .reset_index(drop=True)
)
print(f"closest non-periodic near-misses: showing {len(near_miss_50)} candidates")
display(near_miss_50[SAMPLE_DISPLAY_COLS])
plot_gate_candidate_sample(near_miss_50, "Near-miss sample")


In [ ]:
# Sample 50 non-periodic candidates
non_periodic_sample_50 = sample_gate_candidates(
    df_gate_samples,
    "non_periodic",
    n=SAMPLE_SIZE,
    seed=SAMPLE_BASE_SEED + 2,
)
plot_gate_candidate_sample(non_periodic_sample_50, "Non-periodic sample")


## Smarter Gate Diagnostics

These plots focus on why the deterministic pre-periodicity gate is accepting or rejecting candidates. They complement the phased light-curve panels by showing blocker frequencies, threshold-space structure, and the closest near-misses under the current CE-only support + folded-scatter logic.


In [ ]:
# Smarter periodicity-gate diagnostics (CE-only)
DIAG_LABEL_ORDER = ["periodic", "non_periodic"]
DIAG_LABEL_COLORS = {
    "periodic": "tab:green",
    "non_periodic": "tab:red",
}
DIAG_DISTANCE_COL = "pre_periodicity_edge_distance"
DIAG_ID_COL = next((col for col in ["source_id", "asas_sn_id", "name"] if col in df_gate.columns), None)

df_gate_diag = add_periodicity_edge_metrics(df_gate) if "add_periodicity_edge_metrics" in globals() else df_gate.copy()

diag_thr = _gate_thresholds() if "_gate_thresholds" in globals() else {
    "scatter": float(GATE_KWARGS.get("scatter_ratio_max", 0.8)),
    "ce_snr": float(GATE_KWARGS.get("ce_snr_threshold", 10.0)),
}


def _diag_num(col, default=np.nan):
    if col not in df_gate_diag.columns:
        return pd.Series(default, index=df_gate_diag.index, dtype=float)
    return pd.to_numeric(df_gate_diag[col], errors="coerce")


def _split_tokens(value):
    if pd.isna(value):
        return []
    return [item for item in str(value).split(",") if item]


df_gate_diag["diag_label"] = pd.Categorical(
    df_gate_diag["pre_periodicity_label"],
    categories=DIAG_LABEL_ORDER,
    ordered=True,
)
df_gate_diag["diag_support_count"] = _diag_num("pre_periodicity_support_count", default=0.0).fillna(0.0).astype(int)
df_gate_diag["diag_score"] = _diag_num("pre_periodicity_score")
df_gate_diag["diag_scatter"] = _diag_num("pre_periodicity_scatter_ratio")
df_gate_diag["diag_ce_snr"] = _diag_num("pre_ce_snr")
df_gate_diag["diag_ce_entropy"] = _diag_num("pre_ce_entropy")
df_gate_diag["diag_ce_support"] = df_gate_diag.get("pre_ce_support", pd.Series(False, index=df_gate_diag.index)).fillna(False).astype(bool)
df_gate_diag["diag_alias_flag"] = df_gate_diag.get("pre_periodicity_alias_flag", pd.Series(False, index=df_gate_diag.index)).fillna(False).astype(bool)

df_gate_diag["diag_ce_pass"] = df_gate_diag["diag_ce_snr"].ge(diag_thr["ce_snr"])
df_gate_diag["diag_scatter_pass"] = df_gate_diag["diag_scatter"].le(diag_thr["scatter"])
df_gate_diag["diag_support_pass"] = df_gate_diag["diag_support_count"].ge(1) & df_gate_diag["diag_ce_support"]
df_gate_diag["diag_periodic_logic_pass"] = df_gate_diag["diag_ce_pass"] & df_gate_diag["diag_scatter_pass"]

df_gate_diag["diag_reason_list"] = df_gate_diag.get(
    "pre_periodicity_reason",
    pd.Series("", index=df_gate_diag.index),
).fillna("").map(_split_tokens)


def _diag_blockers_from_row(row):
    if "_gate_blockers" in globals():
        return _gate_blockers(row)

    blockers = []
    if (not np.isfinite(row["diag_ce_snr"])) or row["diag_ce_snr"] < diag_thr["ce_snr"]:
        blockers.append(f"ce[snr<{diag_thr['ce_snr']:g}]")
    if not np.isfinite(row["diag_scatter"]):
        blockers.append("no_folded_scatter")
    elif row["diag_scatter"] > diag_thr["scatter"]:
        blockers.append(f"scatter>{diag_thr['scatter']:g}")
    if row["diag_support_count"] < 1:
        blockers.append("support<1")
    if row["diag_alias_flag"]:
        blockers.append("alias")
    return list(dict.fromkeys(blockers))


df_gate_diag["diag_blocker_list"] = df_gate_diag.apply(_diag_blockers_from_row, axis=1)
df_gate_diag["diag_blocker_count"] = df_gate_diag["diag_blocker_list"].map(len)

diag_rate_cols = {
    "diag_ce_support": "CE support",
    "diag_ce_pass": "CE SNR pass",
    "diag_scatter_pass": "scatter pass",
    "diag_support_pass": "supported candidate",
    "diag_periodic_logic_pass": "periodic logic pass",
    "diag_alias_flag": "alias flag",
}
diag_rate_summary = (
    df_gate_diag.groupby("diag_label")[list(diag_rate_cols)]
    .mean()
    .rename(columns=diag_rate_cols)
    .reindex(DIAG_LABEL_ORDER)
    .round(3)
)
reason_combo_counts = (
    df_gate_diag.get("pre_periodicity_reason", pd.Series("missing", index=df_gate_diag.index))
    .fillna("missing")
    .value_counts()
    .rename_axis("pre_periodicity_reason")
    .to_frame("n")
)

print("pre_periodicity_score is the CE SNR selected by the CE-only deterministic router.")
display(diag_rate_summary)
display(reason_combo_counts.head(15))

reason_tokens = (
    df_gate_diag[["diag_label", "diag_reason_list"]]
    .explode("diag_reason_list")
    .rename(columns={"diag_reason_list": "reason_token"})
)
reason_tokens["reason_token"] = reason_tokens["reason_token"].fillna("none")
top_reason_tokens = reason_tokens["reason_token"].value_counts().head(10).index.tolist()
reason_pivot = (
    reason_tokens[reason_tokens["reason_token"].isin(top_reason_tokens)]
    .groupby(["reason_token", "diag_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=DIAG_LABEL_ORDER, fill_value=0)
)
if top_reason_tokens:
    reason_pivot = reason_pivot.loc[top_reason_tokens]

blocker_tokens = (
    df_gate_diag.loc[df_gate_diag["diag_label"] != "periodic", ["diag_label", "diag_blocker_list"]]
    .explode("diag_blocker_list")
    .rename(columns={"diag_blocker_list": "blocker_token"})
)
blocker_tokens["blocker_token"] = blocker_tokens["blocker_token"].fillna("none")
top_blockers = blocker_tokens["blocker_token"].value_counts().head(12).index.tolist()
blocker_pivot = (
    blocker_tokens[blocker_tokens["blocker_token"].isin(top_blockers)]
    .groupby(["blocker_token", "diag_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=DIAG_LABEL_ORDER, fill_value=0)
)
if top_blockers:
    blocker_pivot = blocker_pivot.loc[top_blockers]

support_mix = pd.crosstab(df_gate_diag["diag_label"], df_gate_diag["diag_support_count"]).reindex(
    index=DIAG_LABEL_ORDER,
    columns=[0, 1],
    fill_value=0,
)
support_mix_norm = support_mix.div(support_mix.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
pass_heat = diag_rate_summary.T

fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
label_colors = [DIAG_LABEL_COLORS[label] for label in DIAG_LABEL_ORDER]

if not reason_pivot.empty:
    reason_pivot.iloc[::-1].plot(kind="barh", stacked=True, ax=axes[0, 0], color=label_colors)
else:
    axes[0, 0].text(0.5, 0.5, "No reason tokens available", ha="center", va="center")
axes[0, 0].set_title("Reason Tokens by Label")
axes[0, 0].set_xlabel("count")
axes[0, 0].grid(axis="x", alpha=0.2)
handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    axes[0, 0].legend(title="label", frameon=False)

if not blocker_pivot.empty:
    blocker_pivot.iloc[::-1].plot(kind="barh", stacked=True, ax=axes[0, 1], color=label_colors)
else:
    axes[0, 1].text(0.5, 0.5, "No blocker tokens available", ha="center", va="center")
axes[0, 1].set_title("Blockers for Non-periodic Candidates")
axes[0, 1].set_xlabel("count")
axes[0, 1].grid(axis="x", alpha=0.2)
handles, labels = axes[0, 1].get_legend_handles_labels()
if handles:
    axes[0, 1].legend(title="label", frameon=False)

support_mix_norm.plot(kind="bar", stacked=True, ax=axes[1, 0], color=["0.80", "0.35"], rot=0)
axes[1, 0].set_title("Support-count Mix Within Each Label")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("fraction")
axes[1, 0].grid(axis="y", alpha=0.2)
axes[1, 0].legend(title="support_count", frameon=False)

im = axes[1, 1].imshow(pass_heat.to_numpy(), aspect="auto", cmap="viridis", vmin=0.0, vmax=1.0)
axes[1, 1].set_xticks(range(len(pass_heat.columns)), pass_heat.columns)
axes[1, 1].set_yticks(range(len(pass_heat.index)), pass_heat.index)
axes[1, 1].set_title("Pass Fraction by Label")
for i, row_name in enumerate(pass_heat.index):
    for j, col_name in enumerate(pass_heat.columns):
        value = pass_heat.loc[row_name, col_name]
        axes[1, 1].text(j, i, f"{value:.2f}", ha="center", va="center", color="white" if value < 0.55 else "black")
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)

for label in DIAG_LABEL_ORDER:
    subset = df_gate_diag[df_gate_diag["diag_label"] == label]
    color = DIAG_LABEL_COLORS[label]
    axes[0, 0].scatter(subset["diag_score"], subset["diag_scatter"], s=18, alpha=0.35, color=color, label=label)
    axes[0, 1].scatter(subset["diag_ce_snr"], subset["diag_ce_entropy"], s=18, alpha=0.35, color=color, label=label)
    axes[1, 0].scatter(subset["pre_periodicity_selected_period"], subset["diag_score"], s=18, alpha=0.35, color=color, label=label)

axes[0, 0].axhline(diag_thr["scatter"], color="0.35", ls="--", lw=1)
axes[0, 0].axvline(diag_thr["ce_snr"], color="0.35", ls=":", lw=1)
axes[0, 0].set_title("CE SNR vs Folded Scatter")
axes[0, 0].set_xlabel("pre_periodicity_score")
axes[0, 0].set_ylabel("folded/raw scatter")
axes[0, 0].grid(alpha=0.2)
axes[0, 0].legend(frameon=False)

axes[0, 1].axvline(diag_thr["ce_snr"], color="0.35", ls="--", lw=1)
axes[0, 1].set_title("CE Entropy vs CE SNR | entropy is diagnostic only")
axes[0, 1].set_xlabel("pre_ce_snr")
axes[0, 1].set_ylabel("pre_ce_entropy")
axes[0, 1].grid(alpha=0.2)
axes[0, 1].legend(frameon=False)

axes[1, 0].axhline(diag_thr["ce_snr"], color="0.35", ls="--", lw=1)
axes[1, 0].set_title("Selected Period vs CE SNR")
axes[1, 0].set_xlabel("pre_periodicity_selected_period")
axes[1, 0].set_ylabel("pre_periodicity_score")
axes[1, 0].grid(alpha=0.2)
axes[1, 0].legend(frameon=False)

if DIAG_DISTANCE_COL in df_gate_diag.columns:
    for label in DIAG_LABEL_ORDER:
        vals = np.sort(df_gate_diag.loc[df_gate_diag["diag_label"] == label, DIAG_DISTANCE_COL].dropna().to_numpy())
        if vals.size == 0:
            continue
        y = np.arange(1, vals.size + 1) / float(vals.size)
        axes[1, 1].plot(vals, y, color=DIAG_LABEL_COLORS[label], lw=2, label=label)
    axes[1, 1].set_title("Periodic-edge Distance CDF")
    axes[1, 1].set_xlabel("pre_periodicity_edge_distance | smaller = closer to periodic")
    axes[1, 1].set_ylabel("CDF")
    axes[1, 1].grid(alpha=0.2)
    axes[1, 1].legend(frameon=False)
else:
    axes[1, 1].text(0.5, 0.5, "Run the edge-metric cell first to plot distance CDFs", ha="center", va="center")
    axes[1, 1].set_axis_off()

plt.show()

near_miss_df = df_gate_diag[df_gate_diag["diag_label"] != "periodic"].copy()
sort_cols = [col for col in [DIAG_DISTANCE_COL, "diag_blocker_count", "diag_score"] if col in near_miss_df.columns]
if sort_cols:
    near_miss_df = near_miss_df.sort_values(sort_cols, ascending=[True, True, False][: len(sort_cols)], na_position="last")
else:
    near_miss_df = near_miss_df.sort_values("diag_score", ascending=False, na_position="last")

near_miss_cols = [
    DIAG_ID_COL,
    "diag_label",
    DIAG_DISTANCE_COL,
    "diag_blocker_count",
    "diag_score",
    "diag_scatter",
    "diag_support_count",
    "diag_ce_snr",
    "diag_ce_entropy",
    "pre_periodicity_reason",
    "diag_blocker_list",
]
near_miss_cols = [col for col in near_miss_cols if col and col in near_miss_df.columns]
print("Closest non-periodic near-misses")
display(near_miss_df[near_miss_cols].head(25))
